# 11.21 Exploration

Exploration methods spend some reward to buy knowledge that can pay back later.

Epsilon-greedy, UCB, and intrinsic bonuses trade short-term reward for information that reduces future regret. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1121
random.seed(SEED)
np.random.seed(SEED)


def build_bandit_ladder():
    ladder = [
        {"name": "D1 2 Bernoulli arms", "means": np.array([0.35, 0.55]), "stds": np.array([0.48, 0.50]), "contextual": False, "nonstationary": False},
        {"name": "D2 3 low-variance arms", "means": np.array([0.42, 0.48, 0.57]), "stds": np.array([0.05, 0.05, 0.05]), "contextual": False, "nonstationary": False},
        {"name": "D3 5 close high-variance arms", "means": np.array([0.48, 0.50, 0.51, 0.52, 0.53]), "stds": np.array([0.28, 0.30, 0.32, 0.30, 0.28]), "contextual": False, "nonstationary": False},
        {"name": "D4 contextual 2-feature bandit", "theta": np.array([[0.2, 0.1], [0.1, 0.4], [0.3, 0.2], [0.0, 0.5]]), "stds": np.array([0.08, 0.08, 0.08, 0.08]), "contextual": True, "nonstationary": False},
        {"name": "D5 nonstationary sparse contextual", "theta": np.array([[0.05, 0.1], [0.4, -0.2], [0.1, 0.35], [0.6, -0.1], [-0.2, 0.55], [0.8, 0.1]]), "stds": np.array([0.2, 0.22, 0.2, 0.25, 0.22, 0.3]), "contextual": True, "nonstationary": True},
    ]
    assert len(ladder) == 5
    assert [len(item.get("means", item.get("theta"))) for item in ladder] == [2, 3, 5, 4, 6]
    return ladder


def context_for_pull(t):
    angle = 0.03 * t
    return np.array([1.0, math.sin(angle)])


def expected_rewards(bandit, t, context):
    if bandit["contextual"]:
        base = bandit["theta"] @ context
        if bandit.get("nonstationary") and t > 120:
            shift = np.linspace(-0.15, 0.20, len(base))
            base = base + shift
        return base
    return bandit["means"]


def sample_reward(bandit, arm, t, context, rng):
    means = expected_rewards(bandit, t, context)
    reward = rng.normal(means[arm], bandit["stds"][arm])
    if bandit.get("nonstationary"):
        reward = float(reward > 0.65)
    return float(reward)


def run_ucb_bandit(bandit, pulls=180, c=1.0, forced=1):
    rng = np.random.default_rng(SEED + len(bandit.get("means", bandit.get("theta"))))
    n_arms = len(bandit.get("means", bandit.get("theta")))
    counts = np.zeros(n_arms, dtype=float)
    sums = np.zeros(n_arms, dtype=float)
    actions = []
    regrets = []
    total_regret = 0.0
    for t in range(1, pulls + 1):
        context = context_for_pull(t)
        means = expected_rewards(bandit, t, context)
        if np.any(counts < forced):
            arm = int(np.argmin(counts))
        else:
            estimates = sums / np.maximum(counts, 1.0)
            bonus = c * np.sqrt(2.0 * np.log(max(t, 2)) / np.maximum(counts, 1.0))
            arm = int(np.argmax(estimates + bonus))
        reward = sample_reward(bandit, arm, t, context, rng)
        counts[arm] = counts[arm] + 1.0
        sums[arm] = sums[arm] + reward
        total_regret = total_regret + float(np.max(means) - means[arm])
        regrets.append(total_regret)
        actions.append(arm)
    values = sums / np.maximum(counts, 1.0)
    return {"counts": counts, "values": values, "actions": actions, "regrets": regrets, "regret": total_regret}


def run_epsilon_bandit(bandit, pulls=180, epsilon=0.1):
    rng = np.random.default_rng(SEED + 7)
    n_arms = len(bandit.get("means", bandit.get("theta")))
    counts = np.zeros(n_arms, dtype=float)
    sums = np.zeros(n_arms, dtype=float)
    total_regret = 0.0
    regrets = []
    for t in range(1, pulls + 1):
        context = context_for_pull(t)
        means = expected_rewards(bandit, t, context)
        if rng.random() < epsilon or counts.sum() == 0:
            arm = int(rng.integers(0, n_arms))
        else:
            arm = int(np.argmax(sums / np.maximum(counts, 1.0)))
        reward = sample_reward(bandit, arm, t, context, rng)
        counts[arm] = counts[arm] + 1.0
        sums[arm] = sums[arm] + reward
        total_regret = total_regret + float(np.max(means) - means[arm])
        regrets.append(total_regret)
    return {"counts": counts, "values": sums / np.maximum(counts, 1.0), "regrets": regrets, "regret": total_regret}


def plot_bandit_panel(ax, result, title):
    ax.bar(np.arange(len(result["counts"])), result["counts"])
    ax.set_title(title)
    ax.set_xlabel("arm")
    ax.set_ylabel("pulls")


## The concept, built once: epsilon and UCB\n\nThe lesson's UCB rule is\n$$a_t=\arg\max_a\left(\hat\mu_a+c\sqrt{\frac{2\ln t}{N_a}}\right).$$\nWe also keep the plan's illustrative $\epsilon=0.1$ exploration probability.

In [ ]:

def select_action_explore(estimates, counts, t, epsilon=0.1, c=1.0, rng=None):
    rng = rng or np.random.default_rng(SEED)
    if rng.random() < epsilon:
        epsilon_action = int(rng.integers(0, len(estimates)))
    else:
        epsilon_action = int(np.argmax(estimates))
    bonus = c * np.sqrt(2.0 * np.log(t) / np.maximum(counts, 1.0))
    ucb_scores = estimates + bonus
    ucb_action = int(np.argmax(ucb_scores))
    return epsilon_action, ucb_action, ucb_scores


estimates = np.array([0.4, 0.5])
counts = np.array([10.0, 5.0])
epsilon_action, ucb_action, ucb_scores = select_action_explore(estimates, counts, t=16, epsilon=0.1, c=1.0, rng=np.random.default_rng(0))
print("ucb scores", np.round(ucb_scores, 6), "chosen", ucb_action)
assert abs(0.1 - 0.1) < 1e-12
assert np.allclose(np.round(ucb_scores, 6), np.array([1.14466, 1.553109]), atol=1e-6)
assert ucb_action == 1


Now use the same selection idea across an arms/variance ladder and add a count-based intrinsic bonus by increasing the UCB coefficient on the hard rung.

In [ ]:

def run_intrinsic_ucb(bandit, pulls=180):
    return run_ucb_bandit(bandit, pulls=pulls, c=1.4, forced=1)


## The dataset ladder (bandit arms/variance ladder)

In [ ]:

ladder = build_bandit_ladder()
for item in ladder:
    arm_count = len(item.get("means", item.get("theta")))
    variance = np.round(item["stds"] ** 2, 4)
    print(item["name"], "arms", arm_count, "contextual", item["contextual"], "variance", variance[:4])


In [ ]:

results = []
artifacts = []
for bandit in ladder:
    result = run_intrinsic_ucb(bandit, pulls=180)
    results.append({"rung": bandit["name"], "regret": result["regret"]})
    artifacts.append((bandit, result))
print("rung | regret")
for row in results:
    print(row["rung"], round(row["regret"], 3))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (bandit, result) in enumerate(artifacts):
    plot_bandit_panel(axes[0, col], result, bandit["name"])
axes[1, 0].plot([idx + 1 for idx in range(len(results))], [row["regret"] for row in results], marker="o")
axes[1, 0].set_title("regret vs rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("cumulative regret")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: ignoring policy support

A rare arm or state cannot be evaluated if it is never sampled. Forced exploration plus a UCB bonus restores support.

In [ ]:

d5 = ladder[-1]
under = run_epsilon_bandit(d5, pulls=180, epsilon=0.0)
fixed = run_ucb_bandit(d5, pulls=180, c=1.4, forced=2)
print("under-explored counts", under["counts"], "regret", round(under["regret"], 3))
print("supported counts", fixed["counts"], "regret", round(fixed["regret"], 3))


## Evaluate it + Practice

- Metric: compare D1-D5 regret against a no-skill baseline.
- Sanity check: D1 should solve by hand and match the exact assertion in the build-once cell.
- Ablation: turn off the key idea and verify the metric worsens on D4 or D5.
- Failure signal: curves that look good only because support, entropy, shape, or rollout bias was hidden.

Practice 1: change one ladder parameter and predict the metric direction before running.


Practice 2: change c from 0.2 to 2.0 and predict which rungs over-explore.